# Notebook 2: Exploratory Data Analysis (EDA)
**NYC Taxi Fare Prediction — Nesha's Work (Part 2: 15 points)**

This notebook performs an in-depth exploration of the dataset to understand data distributions, patterns, and anomalies before modeling.

## 2.1 Setup

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

print('Setup complete')

Setup complete


In [3]:
# Initialize SparkSession
spark = SparkSession.builder \
    .appName('NYC_Taxi_EDA') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark Version: {spark.version}')

Spark Version: 4.1.1


In [ ]:
# Load data from CSV (saved in Notebook 01)
df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv('../data/raw/nyc_taxi_data.csv')

# Convert pickup_datetime to Timestamp
df = df.withColumn(
    'pickup_datetime',
    F.to_timestamp(F.col('pickup_datetime'), 'yyyy-MM-dd HH:mm:ss z')
)

# Ensure plots folder exists
os.makedirs('../data/plots', exist_ok=True)

print(f'Data loaded: {df.count():,} rows, {len(df.columns)} columns')
df.printSchema()

Py4JJavaError: An error occurred while calling o32.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:218)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:132)
	at scala.collection.immutable.List.map(L

## 2.2 Missing Values Analysis

In [ ]:
# Count nulls per column
total = df.count()
null_df = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas().T.reset_index()

null_df.columns = ['Column', 'Null Count']
null_df['Percentage (%)'] = (null_df['Null Count'] / total * 100).round(2)
print('=== MISSING VALUES ===')
print(null_df.to_string(index=False))

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(null_df['Column'], null_df['Percentage (%)'], color='steelblue')
ax.set_xlabel('Null Percentage (%)')
ax.set_title('Missing Values per Column')
for bar, val in zip(bars, null_df['Percentage (%)']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center')
plt.tight_layout()
plt.savefig('../data/plots/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Descriptive Statistics

In [ ]:
# Full descriptive statistics
numeric_cols = ['fare_amount', 'pickup_longitude', 'pickup_latitude',
                'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

stats = df.select(numeric_cols).describe().toPandas()
stats = stats.set_index('summary')
print('=== DESCRIPTIVE STATISTICS ===')
print(stats.to_string())

## 2.4 Fare Amount Distribution (Target Variable)

In [ ]:
# Convert to pandas for visualization
fare_pd = df.select('fare_amount').dropna().toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All data histogram
axes[0].hist(fare_pd['fare_amount'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Fare Amount Distribution (All Data)')
axes[0].set_xlabel('Fare Amount ($)')
axes[0].set_ylabel('Frequency')

# Filtered histogram (0-100$)
fare_filtered = fare_pd[(fare_pd['fare_amount'] > 0) & (fare_pd['fare_amount'] <= 100)]
axes[1].hist(fare_filtered['fare_amount'], bins=100, color='seagreen', edgecolor='white')
axes[1].set_title('Fare Amount Distribution ($0–$100)')
axes[1].set_xlabel('Fare Amount ($)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../data/plots/fare_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Min fare : ${fare_pd["fare_amount"].min():.2f}')
print(f'Max fare : ${fare_pd["fare_amount"].max():.2f}')
print(f'Mean fare: ${fare_pd["fare_amount"].mean():.2f}')
print(f'Median   : ${fare_pd["fare_amount"].median():.2f}')

In [ ]:
# Outlier detection using IQR
q1 = float(df.approxQuantile('fare_amount', [0.25], 0.01)[0])
q3 = float(df.approxQuantile('fare_amount', [0.75], 0.01)[0])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df.filter(
    (F.col('fare_amount') < lower_bound) | (F.col('fare_amount') > upper_bound)
).count()

print(f'Q1: ${q1:.2f} | Q3: ${q3:.2f} | IQR: ${iqr:.2f}')
print(f'Lower bound: ${lower_bound:.2f} | Upper bound: ${upper_bound:.2f}')
print(f'Outlier count: {outliers:,} ({outliers/total*100:.2f}%)')

## 2.5 Passenger Count Distribution

In [ ]:
# Passenger count distribution
passenger_dist = df.groupBy('passenger_count') \
    .count() \
    .orderBy('passenger_count') \
    .toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if (1 <= x <= 6) else 'tomato'
          for x in passenger_dist['passenger_count'].fillna(-1)]
bars = ax.bar(passenger_dist['passenger_count'].astype(str),
              passenger_dist['count'], color=colors, edgecolor='white')
ax.set_title('Passenger Count Distribution')
ax.set_xlabel('Passenger Count')
ax.set_ylabel('Frequency')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 50, f'{int(h):,}',
            ha='center', va='bottom', fontsize=9)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='steelblue', label='Valid (1-6)'),
    plt.Rectangle((0,0),1,1, color='tomato', label='Anomaly')
])
plt.tight_layout()
plt.savefig('../data/plots/passenger_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(passenger_dist.to_string(index=False))

## 2.6 Geographic Coordinates Analysis

In [ ]:
# Valid NYC coordinate bounds
NYC_BOUNDS = {
    'lon_min': -74.3, 'lon_max': -72.9,
    'lat_min':  40.5, 'lat_max':  41.0
}

# Count coordinates outside NYC bounds
invalid_coords = df.filter(
    (F.col('pickup_longitude') < NYC_BOUNDS['lon_min']) |
    (F.col('pickup_longitude') > NYC_BOUNDS['lon_max']) |
    (F.col('pickup_latitude') < NYC_BOUNDS['lat_min']) |
    (F.col('pickup_latitude') > NYC_BOUNDS['lat_max']) |
    (F.col('dropoff_longitude') < NYC_BOUNDS['lon_min']) |
    (F.col('dropoff_longitude') > NYC_BOUNDS['lon_max']) |
    (F.col('dropoff_latitude') < NYC_BOUNDS['lat_min']) |
    (F.col('dropoff_latitude') > NYC_BOUNDS['lat_max'])
).count()

print(f'Coordinates outside NYC bounds: {invalid_coords:,} ({invalid_coords/total*100:.2f}%)')

In [ ]:
# Scatter plot of pickup locations within NYC
coords_pd = df.select('pickup_longitude', 'pickup_latitude') \
    .filter(
        (F.col('pickup_longitude').between(NYC_BOUNDS['lon_min'], NYC_BOUNDS['lon_max'])) &
        (F.col('pickup_latitude').between(NYC_BOUNDS['lat_min'], NYC_BOUNDS['lat_max']))
    ) \
    .limit(10000) \
    .toPandas()

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(coords_pd['pickup_longitude'], coords_pd['pickup_latitude'],
           alpha=0.3, s=5, color='steelblue')
ax.set_title('Pickup Location Distribution (10,000 samples)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('../data/plots/pickup_locations.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Temporal Analysis

In [ ]:
# Extract time features
df_time = df.withColumn('year',        F.year('pickup_datetime')) \
            .withColumn('month',       F.month('pickup_datetime')) \
            .withColumn('hour',        F.hour('pickup_datetime')) \
            .withColumn('day_of_week', F.dayofweek('pickup_datetime'))

# Average fare by hour
fare_by_hour = df_time.groupBy('hour') \
    .agg(F.avg('fare_amount').alias('avg_fare'),
         F.count('*').alias('trip_count')) \
    .orderBy('hour') \
    .toPandas()

# Average fare by day of week
days = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
fare_by_day = df_time.groupBy('day_of_week') \
    .agg(F.avg('fare_amount').alias('avg_fare'),
         F.count('*').alias('trip_count')) \
    .orderBy('day_of_week') \
    .toPandas()
fare_by_day['day_name'] = fare_by_day['day_of_week'].apply(lambda x: days[x-1])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0][0].bar(fare_by_hour['hour'], fare_by_hour['trip_count'],
               color='steelblue', edgecolor='white')
axes[0][0].set_title('Trip Count by Hour')
axes[0][0].set_xlabel('Hour')
axes[0][0].set_ylabel('Trip Count')

axes[0][1].plot(fare_by_hour['hour'], fare_by_hour['avg_fare'],
                marker='o', color='seagreen', linewidth=2)
axes[0][1].set_title('Average Fare by Hour')
axes[0][1].set_xlabel('Hour')
axes[0][1].set_ylabel('Avg Fare ($)')

axes[1][0].bar(fare_by_day['day_name'], fare_by_day['trip_count'],
               color='mediumpurple', edgecolor='white')
axes[1][0].set_title('Trip Count by Day of Week')
axes[1][0].set_xlabel('Day')
axes[1][0].set_ylabel('Trip Count')

axes[1][1].bar(fare_by_day['day_name'], fare_by_day['avg_fare'],
               color='coral', edgecolor='white')
axes[1][1].set_title('Average Fare by Day of Week')
axes[1][1].set_xlabel('Day')
axes[1][1].set_ylabel('Avg Fare ($)')

plt.suptitle('NYC Taxi Temporal Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plots/temporal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.8 Yearly Trend Analysis

In [ ]:
fare_by_year = df_time.groupBy('year') \
    .agg(F.avg('fare_amount').alias('avg_fare'),
         F.count('*').alias('trip_count')) \
    .orderBy('year') \
    .dropna() \
    .toPandas()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(fare_by_year['year'].astype(str), fare_by_year['trip_count'],
        color='steelblue', alpha=0.7, label='Trip Count')
ax2.plot(fare_by_year['year'].astype(str), fare_by_year['avg_fare'],
         color='tomato', marker='o', linewidth=2, label='Avg Fare')

ax1.set_xlabel('Year')
ax1.set_ylabel('Trip Count', color='steelblue')
ax2.set_ylabel('Average Fare ($)', color='tomato')
plt.title('Trip Volume & Fare Trend by Year')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('../data/plots/yearly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.9 Correlation Analysis

In [ ]:
# Compute correlation with fare_amount
print('=== CORRELATION WITH FARE_AMOUNT ===')
for col in ['pickup_longitude', 'pickup_latitude',
            'dropoff_longitude', 'dropoff_latitude', 'passenger_count']:
    corr = df.stat.corr('fare_amount', col)
    print(f'  fare_amount vs {col:25s}: {corr:+.4f}')

In [ ]:
# Correlation heatmap
corr_cols = ['fare_amount', 'pickup_longitude', 'pickup_latitude',
             'dropoff_longitude', 'dropoff_latitude', 'passenger_count']
corr_pd = df.select(corr_cols).dropna().limit(20000).toPandas()

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = corr_pd.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap='coolwarm', center=0, ax=ax,
            xticklabels=[c.replace('_', '\n') for c in corr_cols],
            yticklabels=[c.replace('_', '\n') for c in corr_cols])
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../data/plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.10 EDA Key Findings

| Finding | Details |
|---------|---------|
| Total data | 50,000 rows |
| Missing values | Present in some coordinate & passenger_count columns |
| Fare outliers | Negative values and very large fares exist (need filtering) |
| Passenger anomalies | Values of 0 and >6 need to be removed |
| Invalid coordinates | Some coordinates fall outside NYC bounds |
| Time patterns | Peak hours in morning & evening; higher fares on weekends |
| Yearly trend | Trip volume increased from 2009–2015 |

**Key insight:** Trip distance is likely the strongest predictor of fare, but it's not directly available — it must be calculated from coordinates in the preprocessing stage.

In [ ]:
print('Notebook 02 (EDA) complete!')
# spark.stop()